In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Add the GALFORM python source package
sys.path.append(str(Path("../../src").resolve()))

from galform_analysis.analysis.redshift_space_distortions.subvol_weighted_multipoles import (
    compute_direct_rsd_multipoles,
    compute_weighted_direct_rsd_multipoles,
)
from galform_analysis.utils.read_galaxies import read_galaxy_arrays
from galform_analysis.utils import setconfig
from galform_analysis.config import Cosmology, get_snapshot_redshift

setconfig()
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams.update({
    "figure.figsize": (14, 6),
    "font.size": 11,
    "figure.dpi": 130,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

RESULT_ROOT = Path('../../data/2pcf/ls_wp').resolve()
PLOT_ROOT = Path('./_plots/2pcf_weighted_correction')
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

EPS = 1e-12


def parse_metadata(csv_path):
    rel = csv_path.relative_to(RESULT_ROOT)
    parts = rel.parts
    if len(parts) < 8:
        return None

    sample_name = parts[3]
    seed_match = re.search(r'seed(?P<seed>\d+)', sample_name)

    return {
        'csv_path': csv_path,
        'csv_relpath': str(rel),
        'iz': int(parts[1].removeprefix('iz')),
        'n_subvol': int(parts[6].removeprefix('nsubvol_')),
        'sample_name': sample_name,
        'selection_seed': int(seed_match.group('seed')) if seed_match else -1,
    }



def interp_finite(x, ref_x, ref_y):
    x = np.asarray(x, dtype=float)
    ref_x = np.asarray(ref_x, dtype=float)
    ref_y = np.asarray(ref_y, dtype=float)
    mask = np.isfinite(ref_x) & np.isfinite(ref_y)
    if mask.sum() < 2:
        return np.full_like(x, np.nan, dtype=float)
    return np.interp(x, ref_x[mask], ref_y[mask])



def safe_nanpercentile(values, percentile):
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return np.nan
    return float(np.nanpercentile(values[finite], percentile))


csv_paths = sorted(path for path in RESULT_ROOT.rglob('halo_sampling_convergence_weighted_*.csv') if path.is_file())
if not csv_paths:
    raise FileNotFoundError(f'No 2PCF CSV files found under {RESULT_ROOT}')

frames = []
inventory_rows = []
for csv_path in csv_paths:
    meta = parse_metadata(csv_path)
    if meta is None:
        continue

    df = pd.read_csv(csv_path)
    required = {'r', 'xi_corrected', 'n_subvol'}
    missing = required.difference(df.columns)
    if missing:
        raise KeyError(f'{csv_path} is missing expected columns: {sorted(missing)}')

    df = df.copy()
    for key, value in meta.items():
        df[key] = value
    frames.append(df)

    inventory_rows.append({
        'csv_relpath': meta['csv_relpath'],
        'iz': meta['iz'],
        'n_subvol': meta['n_subvol'],
        'rows': len(df),
        'r_min': float(df['r'].min()),
        'r_max': float(df['r'].max()),
        'seed': meta['selection_seed'],
        'sample_name': meta['sample_name'],
    })

if not frames:
    raise RuntimeError(f'No usable CSV rows were loaded from {RESULT_ROOT}')

raw_data = pd.concat(frames, ignore_index=True)
raw_data['xi_corrected'] = raw_data['xi_corrected'] - 1
raw_data = raw_data.sort_values(['iz', 'n_subvol', 'sample_name', 'r'])
inventory = pd.DataFrame(inventory_rows).sort_values(['iz', 'n_subvol', 'csv_relpath'])

curve_data = (
    raw_data.groupby(['iz', 'n_subvol', 'r'], as_index=False)
    .agg(
        xi_corrected=('xi_corrected', 'mean'),
        xi_std=('xi_corrected', 'std'),
        n_files=('csv_path', 'nunique'),
    )
    .sort_values(['iz', 'n_subvol', 'r'])
)
curve_data['xi_se'] = curve_data['xi_std'] / np.sqrt(curve_data['n_files'])

max_n_by_iz = curve_data.groupby('iz')['n_subvol'].transform('max')
reference_data = curve_data[curve_data['n_subvol'] == max_n_by_iz].copy()

display(inventory.head(20))
print(f'Loaded {len(csv_paths)} CSV files and {len(raw_data):,} total rows')
print(f'Unique redshifts: {sorted(curve_data["iz"].unique().tolist())}')
display(
    inventory.groupby('iz', as_index=False)
    .agg(files=('csv_relpath', 'size'), n_subvols=('n_subvol', 'nunique'), min_n=('n_subvol', 'min'), max_n=('n_subvol', 'max'))
    .sort_values('iz')
)

summary_rows = []
for (iz, n_subvol), subset in curve_data.groupby(['iz', 'n_subvol']):
    ref = reference_data[reference_data['iz'] == iz].sort_values('r')
    subset = subset.sort_values('r')
    r = subset['r'].to_numpy()
    ref_r = ref['r'].to_numpy()
    ref_corrected = interp_finite(r, ref_r, ref['xi_corrected'].to_numpy())
    frac = 100.0 * (subset['xi_corrected'].to_numpy() - ref_corrected) / (np.abs(ref_corrected) + EPS)
    summary_rows.append({
        'iz': iz,
        'n_subvol': int(n_subvol),
        'median_abs_pct_corrected': safe_nanpercentile(np.abs(frac), 50),
        'p90_abs_pct_corrected': safe_nanpercentile(np.abs(frac), 90),
    })

summary = pd.DataFrame(summary_rows).sort_values(['iz', 'n_subvol'])
display(summary)

for iz in sorted(curve_data['iz'].unique().tolist()):
    iz_curve = curve_data[curve_data['iz'] == iz]
    iz_ref = reference_data[reference_data['iz'] == iz].sort_values('r')
    max_n = int(iz_curve['n_subvol'].max())
    n_values = sorted(n for n in iz_curve['n_subvol'].unique().tolist() if n != max_n)
    colors = plt.cm.viridis(np.linspace(0.15, 0.95, max(len(n_values), 1)))

    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(10.5, 8.2), sharex=True, gridspec_kw={'height_ratios': [2.4, 1]})

    ref_r = iz_ref['r'].to_numpy()
    ref_corrected = iz_ref['xi_corrected'].to_numpy()

    for n_subvol, color in zip(n_values, colors):
        subset = iz_curve[iz_curve['n_subvol'] == n_subvol].sort_values('r')
        if subset.empty:
            continue

        r = subset['r'].to_numpy()
        corrected = subset['xi_corrected'].to_numpy()
        se = subset['xi_se'].to_numpy()
        ref_interp = interp_finite(r, ref_r, ref_corrected)

        ax_top.errorbar(r, corrected, yerr=se, fmt='-', lw=2.2, elinewidth=1.0, capsize=3, color=color, label=f'n={n_subvol}')

        valid = np.isfinite(corrected) & np.isfinite(ref_interp) & (np.abs(ref_interp) > EPS)
        if np.any(valid):
            ratio = corrected[valid] / ref_interp[valid]
            ax_bot.plot(r[valid], 100.0 * (ratio - 1.0), lw=1.8, color=color, label=f'n={n_subvol}')

    ax_top.plot(ref_r, ref_corrected, color='black', lw=3.0, ls='-', label=f'n={max_n} (reference)')
    ax_bot.axhline(0.0, color='black', lw=1.5, ls='--')

    ax_top.set_xscale('log')
    ax_top.set_yscale('log')
    ax_bot.set_xscale('log')

    ax_top.set_ylabel(r'$\xi(r)$')
    ax_bot.set_ylabel(r'$100\,(\xi/\xi_{\mathrm{ref}} - 1)$')
    ax_bot.set_xlabel(r'$r\,[h^{-1}\mathrm{Mpc}]$')
    print(iz)
    ax_top.set_title(f'Weighted 2PCF convergence at z={get_snapshot_redshift(str(iz)):.2f}', fontsize=11)
    ax_bot.set_title('Fractional difference to reference', fontsize=10.5)
    ax_top.legend(fontsize=8, ncol=2)
    ax_bot.legend(fontsize=8, ncol=2)

    fig.tight_layout()
    fig.savefig(PLOT_ROOT / f'2pcf_weighted_correction_iz{iz}.png', bbox_inches='tight')
    display(fig)
    plt.close(fig)

print(f'Saved figures to {PLOT_ROOT}')
